In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 62. Week 42 — Observable liquidity and execution boundaries

## 学習目標


- trade-level quoteからquoted/effective/realized spreadを単位付きで計算する。
- FINRA Treasury Daily Aggregateの意味を、個別quote・queue・impactと分けて読む。
- API access/terms gate未通過時に、fixtureの計算を実データ分析へ昇格させない。


## 前提知識


- bid/ask、midpoint、signed tradeの定義
- B11 feasibility noteと、SEC/B9のsource・license・provenance契約

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 62


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

## 1. FINRA access gate

FINRAのDaily Fileは2023-02-13以降の集計を公開しているが、Query APIは認証を要求する。Web画面をscrapeせず、credentials・利用条件・snapshot hash・列定義を先に固定する。現時点の教材はapi_access_granted=Falseであり、Week 42のCore実証は開始しない。

集計に含まれるtrade count、par volume、channel、on/off-the-run、一部VWAPからbid–askやKyle lambdaを逆算してはならない。

In [3]:
api_gate = {
    "api_access_granted": False,
    "terms_reviewed": False,
    "snapshot_sha256": None,
    "real_finra_rows_used": 0,
    "fallback": "trade-level fixture for unit identities only",
}
assert api_gate["real_finra_rows_used"] == 0
aggregate_fields = pd.DataFrame(
    [
        {"field": "dealerCustomerCount", "observable": True, "unit": "trades", "spread_identified": False},
        {"field": "dealerCustomerVolume", "observable": True, "unit": "par value", "spread_identified": False},
        {"field": "volumeWeightedAveragePrice", "observable": True, "unit": "price", "spread_identified": False},
        {"field": "bid_ask_quote", "observable": False, "unit": "not in aggregate", "spread_identified": False},
        {"field": "queue_position", "observable": False, "unit": "not in aggregate", "spread_identified": False},
    ]
)
display(aggregate_fields)

,field,observable,unit,spread_identified
0,dealerCustomerCount,True,trades,False
1,dealerCustomerVolume,True,par value,False
2,volumeWeightedAveragePrice,True,price,False
3,bid_ask_quote,False,not in aggregate,False
4,queue_position,False,not in aggregate,False


In [4]:
bid = np.array([99.90, 99.90, 100.00, 100.00])
ask = np.array([100.10, 100.10, 100.20, 100.20])
trade = np.array([100.10, 99.90, 100.20, 100.00])
future_mid = np.array([100.05, 99.95, 100.10, 100.05])
side = np.array([1.0, -1.0, 1.0, -1.0])
measurements = qt.measure_trade_costs(bid, ask, trade, future_mid, side)
trade_table = pd.DataFrame(
    {
        "quoted_spread": measurements.quoted_spread,
        "effective_spread": measurements.effective_spread,
        "realized_spread": measurements.realized_spread,
        "adverse_selection": measurements.adverse_selection,
    }
)
assert np.all(trade_table["quoted_spread"] >= 0.0)
display(trade_table)
fig = go.Figure()
for column in trade_table.columns:
    fig.add_bar(x=np.arange(len(trade_table)), y=trade_table[column], name=column)
fig.update_layout(
    title="Trade-level identity fixture; not FINRA aggregate evidence",
    xaxis_title="Fixture trade",
    yaxis_title="Price units",
    barmode="group",
    template="plotly_white",
)
fig.show()

,quoted_spread,effective_spread,realized_spread,adverse_selection
0,0.2,0.2,0.1,0.1
1,0.2,0.2,0.1,0.1
2,0.2,0.2,0.2,0.0
3,0.2,0.2,0.1,0.1


## 2. Scenario-only cost

ExecutionCostScenarioはhalf spread、temporary/permanent impact、delay、fundingを明示的な仮定として加算する。これはaggregate dataから推定した値ではない。

## 3. 失敗モード

- FINRA APIを持たないまま公開ページを自動取得する。
- par volumeをspreadへ変換する。
- fixtureのspread identityを実市場の平均costと呼ぶ。
- terms、snapshot、raw columnのprovenanceを残さない。

## 4. 段階別演習

### 基礎

1. buyer/sellerそれぞれでeffective spreadの符号を再計算せよ。

### 標準

2. temporary/permanent impactを別scenarioとしてsensitivity表にせよ。

### 研究

3. API access後のdownload、hash、列schema、利用規約、再配布制限をpre-analysisへ追加せよ。

## 5. Exit Criteria

- [ ] aggregate fieldとtrade-level quoteを分けた
- [ ] spread identityをfixtureで検算した
- [ ] scenario costを推定値と呼ばなかった
- [ ] API/terms gate未通過をreal-data claimへ混ぜなかった

## 6. 出典

- [FINRA Treasury Daily Aggregate Statistics](https://www.finra.org/finra-data/browse-catalog/about-treasury)
- [FINRA Treasury Daily File](https://www.finra.org/finra-data/browse-catalog/about-treasury/daily-file)
- [FINRA Query API](https://developer.finra.org/products/query-api)
- [FINRA Fixed Income Data Specific Terms](https://developer.finra.org/specific-terms-fixed-income-data)